# Legal AI Vietnam - Pipeline on Colab GPU

Chạy toàn bộ pipeline xử lý văn bản pháp luật trên GPU của Google Colab

## Hướng dẫn sử dụng
1. Runtime → Change runtime type → Chọn **T4 GPU**
2. Chạy từng cell theo thứ tự (Shift+Enter)
3. Nhập thông tin xác thực khi được hỏi

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted")

In [ ]:
# Cell 2: Clone repository
import os
import sys

# Clone hoặc cập nhật repository
if not os.path.exists("/content/legal-ai-vn"):
    !git clone https://github.com/makelegalsolutions/legal-ai-vn.git
else:
    %cd legal-ai-vn
    !git pull

%cd /content/legal-ai-vn
print("✅ Repository ready")

In [ ]:
# Cell 3: Tạo service_account.json
from getpass import getpass
import json

print("📝 Nhập nội dung service_account.json:")
print("(Copy từ GitHub Secrets → GDRIVE_SERVICE_ACCOUNT)")

service_json = getpass("Paste JSON content: ")

with open("service_account.json", "w") as f:
    f.write(service_json)

print("✅ service_account.json created")

In [ ]:
# Cell 4: Cài đặt thư viện GPU
!pip install -q streamlit pypdf2 python-docx pydrive2 sentence-transformers google-generativeai beautifulsoup4 requests numpy tqdm python-dotenv gdown rank-bm25 pillow

# Gỡ faiss-cpu, cài faiss-gpu
!pip uninstall -y faiss-cpu -q
!pip install -q faiss-gpu

# Kiểm tra GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU detected! Change runtime to GPU")

In [ ]:
# Cell 5: Thiết lập biến môi trường
from getpass import getpass
import os

print("🔑 Nhập các thông tin xác thực:")

drive_folder_id = getpass("DRIVE_FOLDER_ID (folder laws): ")
faiss_folder_id = getpass("FAISS_DRIVE_FOLDER_ID (folder FAISS): ")
gemini_key = getpass("GEMINI_API_KEY (optional): ")

os.environ["DRIVE_FOLDER_ID"] = drive_folder_id
os.environ["FAISS_DRIVE_FOLDER_ID"] = faiss_folder_id
os.environ["GEMINI_API_KEY"] = gemini_key

print("✅ Environment variables set")

In [ ]:
# Cell 6: Chạy Validation Pipeline
import time

print("=" * 60)
print("STEP 1/4: VALIDATION PIPELINE")
print("=" * 60)

start = time.time()
!python pipeline/validate_pipeline.py
print(f"✅ Completed in {time.time() - start:.2f}s")

In [ ]:
# Cell 7: Chạy Text Processing
print("=" * 60)
print("STEP 2/4: TEXT PROCESSING")
print("=" * 60)

start = time.time()
!python pipeline/text_processing.py
print(f"✅ Completed in {time.time() - start:.2f}s")

In [ ]:
# Cell 8: Chạy Chunking Pipeline
print("=" * 60)
print("STEP 3/4: CHUNKING PIPELINE")
print("=" * 60)

start = time.time()
!python pipeline/chunking_pipeline.py
print(f"✅ Completed in {time.time() - start:.2f}s")

In [ ]:
# Cell 9: Chạy Vector Pipeline (GPU Encoding)
print("=" * 60)
print("STEP 4/4: VECTOR PIPELINE (GPU ENCODING)")
print("=" * 60)

start = time.time()
!python pipeline/vector_pipeline.py
print(f"✅ Completed in {time.time() - start:.2f}s")

In [ ]:
# Cell 10: Kiểm tra kết quả
import json
import os

print("=" * 60)
print("VERIFICATION")
print("=" * 60)

chunks_file = "data/chunks/legal_chunks_latest.json"
if os.path.exists(chunks_file):
    with open(chunks_file, "r") as f:
        chunks = json.load(f)
    print(f"✅ Chunks created: {len(chunks)}")
else:
    print("❌ Chunks file not found")

faiss_file = "data/vectorstore/legal_index.faiss"
if os.path.exists(faiss_file):
    size = os.path.getsize(faiss_file) / 1024 / 1024
    print(f"✅ FAISS index created: {size:.2f} MB")
else:
    print("❌ FAISS file not found")

version_file = "data/vectorstore/latest_version.txt"
if os.path.exists(version_file):
    with open(version_file, "r") as f:
        version = f.read().strip()
    print(f"✅ Latest version: {version}")
else:
    print("❌ Version file not found")

# Kiểm tra file đã upload lên Drive chưa
print("\n📤 Check Google Drive for uploaded FAISS files")

In [ ]:
# Cell 11: Commit lên GitHub (tùy chọn)
from getpass import getpass

commit_choice = input("Commit và push lên GitHub? (y/n): ")

if commit_choice.lower() == 'y':
    github_token = getpass("GitHub Personal Access Token: ")
    github_repo = input("Repository name (makelegalsolutions/legal-ai-vn): ") or "makelegalsolutions/legal-ai-vn"
    
    # Xóa secret file
    !rm -f service_account.json
    
    # Cấu hình git
    !git config user.name "colab-user"
    !git config user.email "colab@colab.com"
    
    # Commit
    !git add data/
    !git commit -m "Colab GPU update: $(date +'%Y-%m-%d %H:%M:%S')"
    
    # Push
    !git push https://{github_token}@github.com/{github_repo}.git HEAD:main
    
    print("✅ Changes pushed to GitHub")
else:
    print("⏭️ Skipped GitHub push")

In [ ]:
# Cell 12: Kết thúc
print("=" * 60)
print("🎉 PIPELINE COMPLETED SUCCESSFULLY!")
print("=" * 60)
print("\n📌 Next steps:")
print("1. Check Google Drive for FAISS files")
print("2. Reboot Streamlit app")
print("3. Test with sample questions")